In [0]:
from pyspark.sql import functions as F

input_path = "/Volumes/workspace/default/cfpb_raw/complaints.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(input_path)
)

df_named = df_raw
for c in df_raw.columns:
    df_named = df_named.withColumnRenamed(c, c.strip().replace(" ", "_"))

df_bronze = (
    df_named
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit(input_path))
    .withColumn("_source_system", F.lit("CFPB Consumer Complaint Database"))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.cfpb_complaints_bronze")
)

df_check = spark.table("workspace.default.cfpb_complaints_bronze")
print(f"Bronze rows: {df_check.count()}")
print(f"Bronze columns: {len(df_check.columns)}")
print(f"Distinct _c0 values: {df_check.select('_c0').distinct().count()}")

Bronze rows: 2023066
Bronze columns: 14
Distinct _c0 values: 2023066


In [0]:
df_check = spark.table("workspace.default.cfpb_complaints_bronze")

total = df_check.count()
distinct_ids = df_check.select("_c0").distinct().count()

print(f"Total rows: {total}")
print(f"Distinct _c0 values: {distinct_ids}")

Total rows: 2023066
Distinct _c0 values: 2023066
